# Упражнение: Сериализация: pipeline, joblib/pickle, SHA-256 и проверка метаданных

## Задание

Дан файл `input.csv` (признаки `age, income, tenure, n_products`, цель `target`, `split`),
реестр сохранённых моделей `registry.csv` и описание окружения сервиса `service_env.json`.

1. Обучите на train `make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))`, передав признаки
   как DataFrame (в указанном порядке столбцов).
2. Сохраните пайплайн двумя способами — `joblib.dump` и `pickle.dump` — во временную папку, загрузите обратно
   и сравните `predict_proba` на test с исходной моделью: `max_diff_joblib`, `max_diff_pickle` — максимум
   |разности| вероятностей класса 1.
3. Метаданные: посчитайте SHA-256 файла joblib до загрузки; `sha_match` = 1, если хеш файла при повторном
   чтении совпал с записанным. Затем допишите в **копию** файла один байт `b'\x00'`:
   `sha_tampered_match` = 1/0 — совпал ли хеш испорченной копии с записанным.
4. `features_match` = 1, если `feature_names_in_` загруженной модели совпадает со списком `features`
   из `service_env.json` (с учётом порядка).
5. `n_changed_without_scaler` — сколько предсказаний (`predict`) на test изменится, если по ошибке сохранить
   и применить только последний шаг пайплайна (LogisticRegression) к **немасштабированным** признакам
   (`pipe[-1].predict(X_test.to_numpy())`).
6. Реестр: модель `compat_<model_id>` = 1, если она совместима с сервисом: совпадают **major.minor** версии
   scikit-learn, **major** версии NumPy и список признаков (с порядком, разделитель `|`). Версии сравнивайте как
   числа, а не как строки.
7. Также выведите `test_accuracy` и `proba_sum` — сумму вероятностей класса 1 по test (контрольное число).

## Подсказки

- Временная папка: `with tempfile.TemporaryDirectory() as d: path = Path(d) / 'model.joblib'`.
  Хеш файла: `hashlib.sha256(path.read_bytes()).hexdigest()`.
- Почему сохраняют весь пайплайн? Последний шаг обучен на стандартизованных признаках; доход ~60 000
  без масштабирования «перевешивает» всё остальное.
- Версию удобно разбирать так: `tuple(int(p) for p in v.split('.')[:2])` — тогда `1.19` ≠ `1.9`,
  а строковое сравнение `'1.19' > '1.9'` дало бы `False`.

## Ожидаемый формат выходного файла

CSV с колонками `check, value` и строками в таком порядке:
`test_accuracy, proba_sum, max_diff_joblib, max_diff_pickle, sha_match, sha_tampered_match, features_match,
n_changed_without_scaler, compat_m1, compat_m2, compat_m3, compat_m4, compat_m5, compat_m6`;
`value` — число, округлённое до 4 знаков.

## Проверка

Сохраните свой результат в `my_answer.csv` (в папке упражнения) и выполните из корня репозитория:

```bash
python tools/check_exercise.py exercises/model_serialization
```

Эталонное решение с пояснениями — `solution.py` (загляните в него только после своей попытки).
Верное решение портал отметит сам и откроет разбор (если он запущен через `python start.py`).

## Данные

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('input.csv')
print(df.shape)
df.head()

## Решение

In [ ]:
# Ваше решение. Итог запишите в DataFrame `result` — в формате, описанном в задании.

result = None

## Проверка

In [ ]:
# Проверка: сравнение result с эталоном (запускайте после своего решения).
# Сохраняет result в my_answer.csv, подсказывает вероятную причину ошибки, верное решение отмечает в портале.
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'tools' / 'check_exercise.py').exists())
sys.path.insert(0, str(root / 'tools'))
from check_exercise import check_result

check_result(result)

---
Эталонное решение с пояснениями — в файле `solution.py`. Загляните в него после своей попытки.